### Atividade Busca Semântica

Nesta atividade você deve aplicar os conhecimentos sobre word embeddings e tokenização para criar um mecanismos de busca semântica. Será disponibilizado um conjunto de dados que possui perguntas médicas, em inglês. Esse conjunto de dados possui as perguntas e também as respostas.

Você deve gerar os vetores das perguntas do conjunto de dados, e permitir que o "usuário" envie a sua pergunta. Você também deve gerar o vetor da pergunta do usuário e com isso buscar a resposta ideal para o usuário. **A resposta ideal é aquela onde o vetor da pergunta do usuário é mais similar ao vetor da pergunta do conjunto de dados** Consulte o notebook "nlp2.ipynb" para verificar como realizamos esse processo

Portanto, no seu script deve ser possível escrever um texto "pergunta" e deve ser retornado a resposta adequada, isto é, a resposta associada a pergunta mais similar no conjunto de dados.

Para isso utilize o pandas e os packages do huggingface

In [23]:
### Download do conjunto de dados

import kagglehub

# Download latest version
path = kagglehub.dataset_download("pythonafroz/medquad-medical-question-answer-for-ai-research")

print("Path to dataset files:", path)

Path to dataset files: /home/codespace/.cache/kagglehub/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/versions/1


In [24]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers.tokenization_utils_base import BatchEncoding

In [25]:
## Amostra do conjunto

df = pd.read_csv(f"{path}/medquad.csv")
df_amostra = df.sample(5000)

In [26]:
df


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
...,...,...,...,...
16407,What is (are) Diabetic Neuropathies: The Nerve...,Focal neuropathy appears suddenly and affects ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16408,How to prevent Diabetic Neuropathies: The Nerv...,The best way to prevent neuropathy is to keep ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16409,How to diagnose Diabetic Neuropathies: The Ner...,Doctors diagnose neuropathy on the basis of sy...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16410,What are the treatments for Diabetic Neuropath...,The first treatment step is to bring blood glu...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...


In [27]:
# Modelo para a língua inglesa

nome_modelo = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
model = AutoModel.from_pretrained(nome_modelo)

In [28]:
## Funções para: 1) Obter os tokens; 2) Obter os embeddings

def get_tokens(pergunta: str) -> str:
    return tokenizer(pergunta, return_tensors="pt")

def get_vetores(tokens_pergunta: BatchEncoding) -> Tensor:

    with torch.no_grad():
        outputs = model(**tokens_pergunta)
        embeddings = outputs.last_hidden_state

    return embeddings

In [29]:
## Novas colunas para os tokens e para os embeddings

df_amostra['tokens'] = df_amostra['question'].apply(lambda x: get_tokens(x))
df_amostra['vetores'] = df_amostra['tokens'].apply(lambda x: get_vetores(x))

In [30]:
def get_cls_embedding(vetores: torch.Tensor) -> torch.Tensor:
    return vetores[0,0,:]

df_amostra['cls_embedding'] = df_amostra['vetores'].apply(get_cls_embedding)

In [31]:
import torch.nn.functional as F

def buscar_resposta(pergunta_usuario: str, df: pd.DataFrame):
    tokens_usuario = get_tokens(pergunta_usuario)
    vetores_usuario = get_vetores(tokens_usuario)
    emb_usuario = get_cls_embedding(vetores_usuario)

    embeddings_dataset = torch.stack(df['cls_embedding'].tolist())

    similaridades = F.cosine_similarity(emb_usuario.unsqueeze(0), embeddings_dataset)

    idx_max = torch.argmax(similaridades).item()

    pergunta_mais_similar = df.iloc[idx_max]['question']
    resposta_ideal = df.iloc[idx_max]['answer']
    score = similaridades[idx_max].item()

    return pergunta_mais_similar, resposta_ideal, score

In [ ]:
pergunta_usuario = input('Digite sua pergunta em inglês: ')
pergunta_similar, resposta, score = buscar_resposta(pergunta_usuario, df_amostra)
print(f'Pergunta mais similar encontrada:\n{pergunta_similar}\n')
print(f'Resposta sugerida:\n{resposta}\n')
print(f'Similaridade: {score:.4f}')

Pergunta mais similar encontrada:
What are the symptoms of Glaucoma ?

Resposta sugerida:
At first, open-angle glaucoma has no symptoms. It causes no pain. Vision seems normal. Without treatment, people with glaucoma will slowly lose their peripheral, or side vision. They seem to be looking through a tunnel. Over time, straight-ahead vision may decrease until no vision remains.

Similaridade: 0.9471
